# Data — analyzer

**Block 3 of 3 in the Data stage.** The other two blocks are scripts that *produce* data; this
notebook is where the data is *understood*. It is also where step 3's third library, the KaxaNuk
**Data Analyzer**, will land: the sections below are the shape that library will fill.

```
Data/curator.py    + Data/Curator/custom_calculations.py    ->  Curator/Time_Series/   m_* + c_*
Data/refinery.py   + Data/Refinery/custom_calculations.py   ->  Refinery/Time_Series/  + r_*
Data/analyzer.ipynb                                          ->  Analyzer/Charts/, the IC table
```

## Division of labour

| Column family | Built by | Scope |
| --- | --- | --- |
| `m_*` | provider, via the Curator | raw market data |
| `c_*` | `Curator/custom_calculations.py` | **per ticker** — one name's own history |
| `r_*` | `Refinery/custom_calculations.py` | **cross-sectional** — names against each other, per date |
| `*_current` | `refinery.py` | joined in from the security master |

## What this notebook is for

`Universe/universe.ipynb` already profiles the *catalogue* — which names exist, which are
delisted, which have no data file — and writes `Universe/Data_Issues.csv`. This notebook does not
repeat that. It looks at the **content**: distributions, regimes, and whether the cross-sectional
layer behaves the way the calculations claim it does (sections 1–5).

It then does the job the Experiments stage depends on: section 6 profiles every candidate feature and
measures how fast each one decays, and **section 7 computes the information coefficient of each feature
against forward returns** — over the whole panel and, once you have declared one, inside the
eligible pool your strategy actually selects from. **A feature that fails there does not get a book
built on it.** That table is how the feature set is chosen from evidence rather than from intuition.

> **Sector caveat.** `sector_current` / `industry_current` are the classification each name carries
> *today*. The provider has no history, so section 8 is a current-snapshot view, and any name
> reclassified inside the window is misattributed before its move.

---

## 0 · Setup

Loads a slim panel: only the columns the analysis below reads, stacked across every refined file.

**Three names in this cell belong to the strategy, and everything else in the notebook is process.**
`ELIGIBILITY_COLUMN` is the 0/1 column that says whether a name may be held — `None` until the
Curator computes one. `SIGNAL_LEVEL_COLUMNS` and `SIGNAL_RANK_COLUMNS` are the candidate features
and their per-date ranks, as the Refinery names them. The template ships with the liquidity rank
only; add a feature to `Data/Refinery/custom_calculations.py`, list its rank here, and sections 5–7
measure it.

In [ ]:
"""Data stage - exploratory analysis over the Curator and Refinery output."""
import pathlib

import matplotlib.pyplot
import numpy
import pandas


def find_repo_root(start):
    """Walk up from `start` to the directory that holds pyproject.toml and Data/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "Data").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(pathlib.Path.cwd())
CURATOR_DIR = REPO_ROOT / "Data" / "Curator" / "Time_Series"
REFINERY_DIR = REPO_ROOT / "Data" / "Refinery" / "Time_Series"
CHART_DIR = REPO_ROOT / "Data" / "Analyzer" / "Charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

# Benchmarks and the cash proxy live in the Curator directory but are not strategy names.
NON_UNIVERSE_TICKERS = frozenset({"BIL", "KN600", "QQQ", "SPY"})

# --- The strategy's columns: the only three names here that are not process ------------------
ELIGIBILITY_COLUMN = None  # e.g. "c_my_signal": 1.0 when a name may be held. None until it exists.
SIGNAL_LEVEL_COLUMNS = []  # raw feature levels, e.g. ["r_momentum_12_1", "r_volatility_63d"]
SIGNAL_RANK_COLUMNS = [    # their per-date ranks; the ranks are what sections 5-7 analyse
    "r_liquidity_rank",
]

PANEL_COLUMNS = [
    "m_date",
    "m_close_dividend_and_split_adjusted",
    "c_daily_traded_value_63d",
    "sector_current",
    "r_return_1d",
    "r_universe_size",
    *([ELIGIBILITY_COLUMN] if ELIGIBILITY_COLUMN else []),
    *SIGNAL_LEVEL_COLUMNS,
    *SIGNAL_RANK_COLUMNS,
]

refinery_paths = sorted(
    path for path in REFINERY_DIR.glob("*.csv") if path.stem not in NON_UNIVERSE_TICKERS
)
assert refinery_paths, f"no refined files in {REFINERY_DIR} - run: uv run python Data/refinery.py"

frames = []
for path in refinery_paths:
    frame = pandas.read_csv(path, usecols=PANEL_COLUMNS, parse_dates=["m_date"])
    frame.insert(0, "ticker", path.stem)
    frames.append(frame)

panel = pandas.concat(frames, ignore_index=True).sort_values(["m_date", "ticker"])
panel["year"] = panel["m_date"].dt.year

print(f"Curator files  : {len(list(CURATOR_DIR.glob('*.csv')))}")
print(f"Refinery files : {len(refinery_paths)} (universe names only)")
print(f"Panel          : {panel['ticker'].nunique()} tickers x {panel['m_date'].nunique()} dates"
      f" = {len(panel):,} rows")
print(f"Window         : {panel['m_date'].min().date()} -> {panel['m_date'].max().date()}")
print(f"Memory         : {panel.memory_usage(deep=True).sum() / 1e6:.0f} MB")
if ELIGIBILITY_COLUMN is None:
    print("\nNo ELIGIBILITY_COLUMN declared: sections 4, 8.1 and the eligible-pool IC are skipped.")

### 0.1 · Chart helpers

In [ ]:
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
SERIES_BLUE = "#2a78d6"
SERIES_ORANGE = "#eb6834"
SERIES_GREEN = "#2f9e6b"
SURFACE = "#fcfcfb"
GRID = "#ebeae5"

matplotlib.pyplot.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": "#d8d7d2",
    "axes.labelcolor": INK_SECONDARY,
    "text.color": INK_PRIMARY,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "font.size": 10,
    "axes.titlesize": 12,
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
})


def style_axes(axes, title=None, subtitle=None, ylabel=None, xlabel=None):
    """Common furniture: left-aligned bold title, optional subtitle line, recessive grid."""
    if title:
        axes.set_title(title, loc="left", pad=24 if subtitle else 8, weight="bold")
    if subtitle:
        axes.annotate(
            subtitle, xy=(0, 1), xycoords="axes fraction",
            xytext=(0, 6), textcoords="offset points",
            fontsize=9, color=INK_SECONDARY, va="bottom", ha="left",
        )
    if ylabel:
        axes.set_ylabel(ylabel)
    if xlabel:
        axes.set_xlabel(xlabel)
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right"):
        axes.spines[side].set_visible(False)
    return axes


def save(figure, file_name):
    """Write a figure to the analyzer chart directory and show it."""
    figure.tight_layout()
    figure.savefig(CHART_DIR / file_name)
    matplotlib.pyplot.show()


print("Chart helpers ready.")

---

## 1 · Column inventory — what each stage contributed

The refined file is the Curator file plus columns, same rows. Confirming that here means every
later section can read one directory and forget the Curator exists.

In [ ]:
sample_ticker = refinery_paths[0].stem
curator_columns = list(pandas.read_csv(CURATOR_DIR / f"{sample_ticker}.csv", nrows=0).columns)
refinery_columns = list(pandas.read_csv(REFINERY_DIR / f"{sample_ticker}.csv", nrows=0).columns)

added = [column for column in refinery_columns if column not in curator_columns]
dropped = [column for column in curator_columns if column not in refinery_columns]

inventory = pandas.DataFrame({
    "family": ["m_* (provider)", "c_* (Curator)", "r_* (Refinery)", "*_current (join)"],
    "columns": [
        len([column for column in refinery_columns if column.startswith("m_")]),
        len([column for column in refinery_columns if column.startswith("c_")]),
        len([column for column in refinery_columns if column.startswith("r_")]),
        len([column for column in refinery_columns if column.endswith("_current")]),
    ],
})
print(inventory.to_string(index=False))
print(f"\nCurator columns : {len(curator_columns)}")
print(f"Refinery columns: {len(refinery_columns)}")
carried = "none - the Curator output is carried through intact"
print(f"Dropped by the refinery: {dropped if dropped else carried}")
print(f"\nAdded ({len(added)}):")
for column in added:
    print(f"  {column}")

curator_rows = len(pandas.read_csv(CURATOR_DIR / f"{sample_ticker}.csv", usecols=["m_date"]))
refinery_rows = len(pandas.read_csv(REFINERY_DIR / f"{sample_ticker}.csv", usecols=["m_date"]))
print(f"\nRow counts for {sample_ticker}: curator {curator_rows}, refinery {refinery_rows}"
      f" -> {'aligned' if curator_rows == refinery_rows else 'MISALIGNED'}")

---

## 2 · Returns — what the price data actually looks like

The daily total return is the series every downstream P&L number is built on. Two things matter:
how fat the tails are (which decides whether a mean/variance summary is misleading), and whether
any return is large enough to be a data error rather than a market move.

In [ ]:
returns = panel["r_return_1d"].dropna()

print(f"observations   : {len(returns):,}")
print(f"mean / median  : {returns.mean():.5f} / {returns.median():.5f}")
print(f"std dev (daily): {returns.std():.4f}  -> annualised {returns.std() * numpy.sqrt(252):.1%}")
print(f"skew / kurtosis: {returns.skew():.2f} / {returns.kurtosis():.1f}"
      "   (a normal distribution has 0 / 0)")
for quantile in (0.001, 0.01, 0.5, 0.99, 0.999):
    print(f"  q{quantile:<6} {returns.quantile(quantile):+.4f}")

# --- Distribution against a normal with the same mean and standard deviation ------------
figure, axes = matplotlib.pyplot.subplots(1, 2, figsize=(13, 4.2))
clipped = returns.clip(-0.25, 0.25)
axes[0].hist(clipped, bins=200, color=SERIES_BLUE, density=True)
grid = numpy.linspace(-0.25, 0.25, 400)
normal = numpy.exp(-0.5 * ((grid - returns.mean()) / returns.std()) ** 2) / (
    returns.std() * numpy.sqrt(2 * numpy.pi)
)
axes[0].plot(grid, normal, color=SERIES_ORANGE, linewidth=1.6, label="normal, same mean/sd")
axes[0].legend(frameon=False, fontsize=9)
axes[0].set_yscale("log")
style_axes(
    axes[0], "Daily return distribution (log density)",
    subtitle="clipped at +/-25% for display; the log scale is what makes the tails visible",
    xlabel="daily total return",
)

extreme = returns[returns.abs() > 0.5]
axes[1].hist(returns[returns.abs() > 0.2], bins=80, color=SERIES_ORANGE)
style_axes(
    axes[1], "The tail beyond +/-20%",
    subtitle=f"{len(returns[returns.abs() > 0.2]):,} observations; {len(extreme):,} beyond +/-50%",
    xlabel="daily total return", ylabel="observations",
)
save(figure, "returns_distribution.png")

print(f"\nMoves beyond +/-50%: {len(extreme)}")
if len(extreme):
    worst = panel.loc[returns.abs().sort_values(ascending=False).head(10).index]
    print(worst[["m_date", "ticker", "r_return_1d"]].to_string(index=False))
    print("  -> check these against splits/corporate actions before trusting them as returns.")

### 2.1 · Dispersion over time

Cross-sectional dispersion — how far apart the names move on a given day — is what a
stock-selection strategy has to work with. A flat market where everything moves together offers a
selection strategy nothing, however good the signal is.

In [ ]:
daily = panel.groupby("m_date").agg(
    dispersion=("r_return_1d", "std"),
    mean_return=("r_return_1d", "mean"),
    names=("ticker", "size"),
)
rolling_dispersion = daily["dispersion"].rolling(63).mean()

figure, axes = matplotlib.pyplot.subplots(figsize=(12, 4.2))
axes.plot(daily.index, daily["dispersion"], color=GRID, linewidth=0.6)
axes.plot(rolling_dispersion.index, rolling_dispersion, color=SERIES_BLUE, linewidth=1.8)
style_axes(
    axes, "Cross-sectional return dispersion",
    subtitle="daily standard deviation across names (grey) and its 63-day average (blue)",
    ylabel="std dev of daily returns across names",
)
save(figure, "return_dispersion.png")

print("Median dispersion by year:")
print(panel.groupby("year")["r_return_1d"].std().round(4).to_string())

---

## 3 · Liquidity — the column that decides capacity

Whatever the strategy sizes by, average daily traded value decides how much of it can be run.
Traded value spans orders of magnitude, so it is shown on a log scale; the concentration curve
then answers the question that matters for capacity: how much of the universe's total traded value
sits in the top handful of names. If the strategy *weights* by liquidity, this distribution is the
shape of the portfolio.

In [ ]:
latest_date = panel["m_date"].max()
latest = panel[panel["m_date"] == latest_date].dropna(subset=["c_daily_traded_value_63d"])
traded_value = latest["c_daily_traded_value_63d"].sort_values(ascending=False)

figure, axes = matplotlib.pyplot.subplots(1, 2, figsize=(13, 4.2))
axes[0].hist(numpy.log10(traded_value[traded_value > 0]), bins=50, color=SERIES_BLUE)
style_axes(
    axes[0], "63-day average traded value",
    subtitle=f"{latest_date.date()}, {len(traded_value)} names",
    xlabel="log10 USD per day", ylabel="names",
)

cumulative = traded_value.cumsum() / traded_value.sum()
axes[1].plot(range(1, len(cumulative) + 1), cumulative.to_numpy(), color=SERIES_BLUE, linewidth=2)
for count in (10, 50, 100):
    if count <= len(cumulative):
        share = cumulative.iloc[count - 1]
        axes[1].plot([count], [share], "o", markersize=7, color=SERIES_ORANGE,
                     markeredgecolor=SURFACE, markeredgewidth=2, zorder=3)
        axes[1].annotate(f"top {count} = {share:.0%}", xy=(count, share),
                         xytext=(8, -12), textcoords="offset points",
                         fontsize=9, color=INK_SECONDARY)
axes[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
style_axes(
    axes[1], "Concentration of traded value",
    subtitle="cumulative share, names ranked by ADTV",
    xlabel="names, ranked", ylabel="cumulative share of total traded value",
)
save(figure, "liquidity_distribution.png")

print(f"Median ADTV : ${traded_value.median():,.0f}/day")
print(f"Top 10 share: {cumulative.iloc[9]:.1%} of all traded value in the universe")

---

## 4 · Regime — the market state the eligibility rule reacts to

**Breadth** — the share of the cross-section that is eligible on each date — is the same value for
every name, so it is a market reading rather than a per-name one, and it is the single most useful
diagnostic a threshold signal has: it says whether the strategy holds many names because many are
genuinely qualifying, or because the whole market is. Runs only once `ELIGIBILITY_COLUMN` is set.

In [ ]:
if ELIGIBILITY_COLUMN is None:
    print("Skipped: no ELIGIBILITY_COLUMN declared in section 0.")
else:
    breadth = panel.groupby("m_date")[ELIGIBILITY_COLUMN].mean().dropna()

    figure, axes = matplotlib.pyplot.subplots(figsize=(12, 4.4))
    axes.fill_between(breadth.index, breadth.to_numpy(), color=SERIES_BLUE, alpha=0.22)
    axes.plot(breadth.index, breadth.to_numpy(), color=SERIES_BLUE, linewidth=1.4)
    axes.axhline(0.5, color=INK_SECONDARY, linewidth=1, linestyle="--")
    axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    axes.set_ylim(0, 1)
    style_axes(
        axes, f"Breadth: share of the universe with {ELIGIBILITY_COLUMN} == 1",
        subtitle="the dashed line is half the universe",
        ylabel="share of names",
    )
    save(figure, "breadth_over_time.png")

    yearly_breadth = panel.groupby("year")[ELIGIBILITY_COLUMN].mean().dropna()
    print("Mean breadth by year:")
    for year, value in yearly_breadth.items():
        bar = "#" * int(round(value * 50))
        print(f"  {year}  {value:.1%}  {bar}")

### 4.1 · Signal persistence

A threshold strategy pays transaction costs every time a name flips. How long a name stays on the
right side of its own rule therefore sets the floor on turnover, before any portfolio rule is
applied.

In [ ]:
if ELIGIBILITY_COLUMN is None:
    print("Skipped: no ELIGIBILITY_COLUMN declared in section 0.")
else:
    signal = panel.pivot_table(
        index="m_date", columns="ticker", values=ELIGIBILITY_COLUMN, aggfunc="first",
    )
    flips = (signal.diff().abs() == 1)
    daily_flip_rate = flips.sum(axis=1) / signal.notna().sum(axis=1)

    # Mean run length: how many trading days a name holds one state before switching.
    runs = []
    for ticker in signal.columns:
        series = signal[ticker].dropna()
        if len(series) < 2:
            continue
        changes = (series != series.shift()).cumsum()
        runs.extend(series.groupby(changes).size().tolist())
    runs = pandas.Series(runs)

    figure, axes = matplotlib.pyplot.subplots(1, 2, figsize=(13, 4.2))
    axes[0].plot(daily_flip_rate.index, daily_flip_rate.rolling(21).mean(), color=SERIES_BLUE)
    axes[0].yaxis.set_major_formatter(lambda value, _: f"{value:.1%}")
    style_axes(
        axes[0], "Share of names flipping eligibility each day",
        subtitle="21-day average", ylabel="share of names",
    )
    axes[1].hist(runs.clip(upper=500), bins=60, color=SERIES_BLUE)
    axes[1].axvline(runs.median(), color=SERIES_ORANGE, linewidth=1.6)
    style_axes(
        axes[1], "How long an eligibility state lasts",
        subtitle=f"median {runs.median():.0f} trading days (orange); clipped at 500 for display",
        xlabel="consecutive trading days in one state", ylabel="runs",
    )
    save(figure, "signal_persistence.png")

    print(f"Runs            : {len(runs):,}")
    print(f"Median run length: {runs.median():.0f} trading days")
    print(f"Runs of 21 days or fewer: {(runs <= 21).mean():.1%}  <- these are the whipsaws")

---

## 5 · Do the cross-sectional columns behave as claimed?

The Refinery asserts two properties. Both are checkable, and both would be silent if broken.

1. **Ranks are per-date percentiles.** Pooled across the sample they must be near-uniform on
   [0, 1] and each date's mean must sit at ~0.5, whatever the number of names that day.
2. **Nothing leaks across time.** A rank computed over the pooled sample instead of per date would
   show a drifting per-date mean, because the universe's composition changes.

In [ ]:
column_count = len(SIGNAL_RANK_COLUMNS)
figure, axes_grid = matplotlib.pyplot.subplots(
    1, column_count, figsize=(3.6 * column_count + 1, 3.6), squeeze=False,
)
for axes, column in zip(axes_grid.flatten(), SIGNAL_RANK_COLUMNS):
    axes.hist(panel[column].dropna(), bins=50, color=SERIES_BLUE)
    axes.set_ylim(bottom=0)
    style_axes(axes, column.replace("r_", "").replace("_rank", ""), xlabel="percentile")
figure.suptitle(
    "Pooled rank distributions - flat is correct", x=0.01, ha="left", fontsize=13, weight="bold",
)
figure.tight_layout(rect=(0, 0, 1, 0.9))
figure.savefig(CHART_DIR / "rank_distributions.png")
matplotlib.pyplot.show()

daily_rank_mean = panel.groupby("m_date")[SIGNAL_RANK_COLUMNS].mean()
figure, axes = matplotlib.pyplot.subplots(figsize=(12, 4.2))
for column in SIGNAL_RANK_COLUMNS:
    axes.plot(daily_rank_mean.index, daily_rank_mean[column], linewidth=1.0, label=column)
axes.axhline(0.5, color=INK_SECONDARY, linewidth=1, linestyle="--")
axes.set_ylim(0.3, 0.7)
axes.legend(frameon=False, fontsize=8, ncol=4)
style_axes(
    axes, "Per-date mean of each rank",
    subtitle="a per-date percentile pins this at 0.5;"
             " drift would mean the rank leaked across dates",
    ylabel="mean rank",
)
save(figure, "rank_causality_check.png")

print("Per-date rank means (pooled over all dates):")
print(daily_rank_mean.mean().round(4).to_string())
print("\nMax deviation of any date's mean from 0.5:")
print((daily_rank_mean - 0.5).abs().max().round(4).to_string())
print("\n-> values at 0.5 with tiny deviation confirm the ranks are per-date, not pooled.")

---

## 6 · The signal library — what each candidate feature looks like

Before any feature is traded it is worth knowing three things: what its distribution looks like,
how much of the panel it covers, and **how fast it decays** — the rank autocorrelation at a lag is
what decides whether a feature can be traded at that horizon or whether it will simply generate
turnover.

**The template ships no candidate features.** Add each one to `Data/Refinery/custom_calculations.py`
as a level and a per-date rank, list both in section 0, and record it here:

| Feature | Reads as | Role |
| --- | --- | --- |
| <`r_feature`> | <one sentence> | <selection, weighting, exit anchor, or diagnostic> |

In [ ]:
if not SIGNAL_LEVEL_COLUMNS:
    print("No SIGNAL_LEVEL_COLUMNS declared in section 0 - nothing to profile yet.")
else:
    column_count = len(SIGNAL_LEVEL_COLUMNS)
    figure, axes_grid = matplotlib.pyplot.subplots(
        1, column_count, figsize=(4.4 * column_count + 1, 3.8), squeeze=False,
    )
    for axes, column in zip(axes_grid.flatten(), SIGNAL_LEVEL_COLUMNS):
        values = panel[column].dropna()
        # Trimmed for display only: long-tailed features would otherwise compress into one bin.
        lower, upper = values.quantile([0.005, 0.995])
        axes.hist(values.clip(lower, upper), bins=60, color=SERIES_BLUE)
        style_axes(
            axes, column.replace("r_", ""),
            subtitle=f"median {values.median():.4f}"
                     f" | coverage {values.notna().sum() / len(panel):.0%}",
        )
    figure.suptitle(
        "Candidate feature distributions (0.5-99.5 percentile shown)",
        x=0.01, ha="left", fontsize=13, weight="bold",
    )
    figure.tight_layout(rect=(0, 0, 1, 0.9))
    figure.savefig(CHART_DIR / "signal_distributions.png")
    matplotlib.pyplot.show()

    inventory = pandas.DataFrame({
        "coverage": [panel[column].notna().mean() for column in SIGNAL_LEVEL_COLUMNS],
        "median": [panel[column].median() for column in SIGNAL_LEVEL_COLUMNS],
        "p05": [panel[column].quantile(0.05) for column in SIGNAL_LEVEL_COLUMNS],
        "p95": [panel[column].quantile(0.95) for column in SIGNAL_LEVEL_COLUMNS],
    }, index=SIGNAL_LEVEL_COLUMNS)
    print("Feature levels across the whole panel:")
    print(inventory.round(4).to_string())

### 6.1 · Signal decay — how long a rank stays put

The autocorrelation of a feature's **rank** at lag *k* answers the question that decides turnover:
if I select on this feature today, how much of that selection still holds *k* days from now? A
feature whose rank autocorrelation collapses within a month cannot drive selection in a book that
rebalances on composition changes — it would trigger constantly and pay costs for noise.

A fast-decaying feature is not useless, it is just an *exit* signal rather than an entry one.

In [ ]:
DECAY_LAGS = (1, 5, 21, 63, 252)

# Ranks are already per-date uniform, so a plain Pearson correlation between a rank today and the
# same rank k days later is the rank autocorrelation. Computed on the wide matrix so the shift is
# per ticker rather than across the stacked panel.
decay_rows = {}
for column in SIGNAL_RANK_COLUMNS:
    wide = panel.pivot_table(index="m_date", columns="ticker", values=column, aggfunc="first")
    decay_rows[column] = {
        f"lag {lag}d": wide.corrwith(wide.shift(lag), axis=0).mean()
        for lag in DECAY_LAGS
    }

decay = pandas.DataFrame(decay_rows).T
print("Rank autocorrelation (mean across names):")
print(decay.round(3).to_string())

figure, axes = matplotlib.pyplot.subplots(figsize=(9, 4.6))
for column in decay.index:
    axes.plot(
        DECAY_LAGS, decay.loc[column].to_numpy(), marker="o", markersize=4,
        linewidth=1.5, label=column.replace("r_", "").replace("_rank", ""),
    )
axes.axhline(0, color=INK_SECONDARY, linewidth=1)
axes.set_xscale("log")
axes.set_xticks(DECAY_LAGS)
axes.set_xticklabels([f"{lag}d" for lag in DECAY_LAGS])
axes.legend(frameon=False, fontsize=8, ncol=2)
style_axes(
    axes, "How fast does each feature's rank decay?",
    subtitle="rank autocorrelation vs lag; a flat line is a slow, tradeable feature",
    ylabel="rank autocorrelation", xlabel="lag (trading days)",
)
save(figure, "signal_decay.png")

print("\nReading: a feature still above ~0.5 at lag 21d survives a monthly holding period;")
print("one that has fallen near 0 by then is generating turnover rather than persistent selection.")

---

## 7 · Information coefficient — which features actually predict returns

The decisive table for every experiment after the benchmark. For each feature and each horizon, the
**information coefficient** is the cross-sectional Spearman correlation between the feature's
rank on date *t* and the forward return from *t* to *t+h*, computed **per date** and then
averaged. Per date is what keeps it causal: the correlation only ever compares names that were
observable at the same moment.

Three numbers per pair:

- **IC** — the mean daily correlation. Sign matters as much as size: a negative IC means the
  feature works *inverted* (low volatility outperforming, for example).
- **IC std** — how much the relationship moves around.
- **IR** = IC / IC std — the consistency of the edge, which is what survives into a portfolio.
  In equities an IC of 0.02–0.05 in absolute value is a normal, usable signal; anything above
  0.10 deserves suspicion of look-ahead before celebration.

Computed over the **whole panel**, and — once `ELIGIBILITY_COLUMN` is set — over the **eligible pool
only**. The second is the one that matters, because that pool is what the strategy actually selects
from, and a feature can behave differently inside an already-filtered group. **The IC table is a
screening tool, not evidence**: a feature that passes here has earned a backtest, not a belief.

In [ ]:
IC_HORIZONS = (21, 63, 252)


def add_forward_returns(frame, horizons):
    """
    Forward total return over each horizon, computed within each ticker.

    Grouping by ticker is what stops one name's last rows reaching into the next name's first
    ones. The shift is negative, so row t carries the return earned *after* t - which is
    precisely the thing a signal observed at t is being asked to predict.
    """
    ordered = frame.sort_values(["ticker", "m_date"])
    price = ordered["m_close_dividend_and_split_adjusted"]
    grouped = price.groupby(ordered["ticker"])
    forward = {
        f"forward_{horizon}d": (grouped.shift(-horizon) / price) - 1.0
        for horizon in horizons
    }

    return pandas.DataFrame(forward)


def cross_sectional_ic(frame, signal_column, forward_column, date_column="m_date"):
    """
    Mean per-date Spearman IC, its dispersion, and the implied information ratio.

    Both series are re-ranked inside each date and correlated there, so this is a Spearman
    correlation computed one cross-section at a time - never pooled across dates, which would let
    the sample's own time trend masquerade as predictive power.
    """
    usable = frame[[date_column, signal_column, forward_column]].dropna()
    if len(usable) == 0:
        return {"ic": numpy.nan, "ic_std": numpy.nan, "ir": numpy.nan, "dates": 0}

    grouped = usable.groupby(date_column)
    signal_rank = grouped[signal_column].rank(pct=True)
    forward_rank = grouped[forward_column].rank(pct=True)

    frame_ranked = pandas.DataFrame({
        date_column: usable[date_column],
        "signal": signal_rank,
        "forward": forward_rank,
    })
    by_date = frame_ranked.groupby(date_column)
    signal_deviation = frame_ranked["signal"] - by_date["signal"].transform("mean")
    forward_deviation = frame_ranked["forward"] - by_date["forward"].transform("mean")

    products = (signal_deviation * forward_deviation).groupby(frame_ranked[date_column]).sum()
    signal_energy = (signal_deviation ** 2).groupby(frame_ranked[date_column]).sum()
    forward_energy = (forward_deviation ** 2).groupby(frame_ranked[date_column]).sum()
    denominator = numpy.sqrt(signal_energy * forward_energy)
    daily_ic = (products / denominator.where(denominator > 0)).dropna()

    return {
        "ic": daily_ic.mean(),
        "ic_std": daily_ic.std(),
        "ir": daily_ic.mean() / daily_ic.std() if daily_ic.std() > 0 else numpy.nan,
        "dates": len(daily_ic),
    }


forward_returns = add_forward_returns(panel, IC_HORIZONS)
panel_with_forward = panel.join(forward_returns)

scopes = [("all names", panel_with_forward)]
if ELIGIBILITY_COLUMN is not None:
    eligible_pool = panel_with_forward[panel_with_forward[ELIGIBILITY_COLUMN] == 1.0]
    scopes.append(("eligible pool", eligible_pool))
    print(f"Whole panel  : {len(panel_with_forward):,} rows")
    print(f"Eligible pool: {len(eligible_pool):,} rows"
          f" ({len(eligible_pool) / len(panel_with_forward):.0%})\n")

ic_rows = []
for scope_name, scope in scopes:
    for signal_column in SIGNAL_RANK_COLUMNS:
        for horizon in IC_HORIZONS:
            statistics = cross_sectional_ic(scope, signal_column, f"forward_{horizon}d")
            ic_rows.append({
                "scope": scope_name,
                "signal": signal_column.replace("r_", "").replace("_rank", ""),
                "horizon": f"{horizon}d",
                **statistics,
            })

ic_table = pandas.DataFrame(ic_rows)
ic_pivot = ic_table.pivot_table(
    index="signal", columns=["scope", "horizon"], values="ic",
).round(4)
ir_pivot = ic_table.pivot_table(
    index="signal", columns=["scope", "horizon"], values="ir",
).round(3)

print("Information coefficient (mean per-date Spearman rank correlation with forward return):")
print(ic_pivot.to_string())
print("\nInformation ratio (IC / IC std - the consistency of the edge):")
print(ir_pivot.to_string())

ic_path = CHART_DIR.parent / "signal_information_coefficients.csv"
ic_table.to_csv(ic_path, index=False)
print(f"\nWritten: {ic_path.relative_to(REPO_ROOT)}")

### 7.1 · Reading the IC table

The chart puts the decisive scope's ICs side by side, which is how a feature set should be chosen:
**sign first** (does the feature work as stated, or inverted?), **then consistency**, and only then
size. A prediction made here *before* a backtest — "this weighting will cost return because the IC
has the wrong sign" — and confirmed by the backtest afterwards is the strongest methodological
result an experiment can report.

In [ ]:
decisive_scope = scopes[-1][0]
scope_ic = ic_table[ic_table["scope"] == decisive_scope]
signals_ordered = (
    scope_ic.groupby("signal")["ic"].apply(lambda values: values.abs().max())
    .sort_values(ascending=False).index
)

figure, axes = matplotlib.pyplot.subplots(figsize=(10, max(2.4, 0.9 * len(signals_ordered) + 1.6)))
bar_height = 0.26
positions = numpy.arange(len(signals_ordered))
for offset, horizon in zip((-bar_height, 0.0, bar_height), (f"{h}d" for h in IC_HORIZONS)):
    values = [
        scope_ic.loc[
            (scope_ic["signal"] == signal) & (scope_ic["horizon"] == horizon), "ic"
        ].squeeze()
        for signal in signals_ordered
    ]
    axes.barh(positions + offset, values, height=bar_height, label=horizon)

axes.set_yticks(positions)
axes.set_yticklabels(signals_ordered)
axes.axvline(0, color=INK_SECONDARY, linewidth=1.2)
axes.legend(frameon=False, fontsize=9, title="forward horizon", ncol=3)
style_axes(
    axes, f"Information coefficient, {decisive_scope}",
    subtitle="positive = a high rank predicts a high forward return;"
             " negative = the feature works inverted",
    xlabel="mean per-date Spearman IC",
)
axes.spines["left"].set_visible(False)
save(figure, "signal_information_coefficient.png")

best = scope_ic.reindex(scope_ic["ic"].abs().sort_values(ascending=False).index).head(6)
print(f"Strongest signal/horizon pairs ({decisive_scope}):")
print(best[["signal", "horizon", "ic", "ir", "dates"]].to_string(index=False))
print("\nA negative IC is not a failure - it means the feature is used inverted.")

---

## 8 · Sector view *(current-snapshot only)*

Sector is joined in from the security master and is the classification each name carries **today**.
It is shown because sector composition of the tradeable set is worth knowing, and flagged because
using it for attribution before a reclassification date would be wrong. Any sector-relative feature
carries the same caveat and is a diagnostic column, never a selection input.

In [ ]:
with_sector = panel.dropna(subset=["sector_current"])

if ELIGIBILITY_COLUMN is not None:
    sector_breadth = (
        with_sector.groupby(["year", "sector_current"])[ELIGIBILITY_COLUMN].mean().unstack()
    )
    print("Share of each sector eligible, by year (current classification):")
    print((sector_breadth * 100).round(0).astype("Int64").to_string())

latest_sector = with_sector[with_sector["m_date"] == with_sector["m_date"].max()]
counts = latest_sector["sector_current"].value_counts().sort_values()

figure, axes = matplotlib.pyplot.subplots(figsize=(9, 4.6))
axes.barh(counts.index.astype(str), counts.to_numpy(), color=SERIES_BLUE, height=0.72)
for position, value in enumerate(counts.to_numpy()):
    axes.text(value + counts.max() * 0.012, position, str(value),
              va="center", fontsize=9, color=INK_SECONDARY)
axes.set_xlim(0, counts.max() * 1.15)
style_axes(
    axes, "Names per sector with data on the latest date",
    subtitle="classification as of today - not point-in-time", xlabel="names",
)
axes.spines["left"].set_visible(False)
save(figure, "sector_composition.png")

---

## 9 · Handoff

What the Data stage produces and what an experiment consumes.

| Output | Consumed by |
| --- | --- |
| `Data/Curator/Time_Series/` | the Refinery (and anything wanting raw `m_*` / `c_*`) |
| `Data/Curator/Benchmarks/` | backtest and attribution |
| `Data/Refinery/Time_Series/` | **`Experiments/` — read this one** |
| `Data/Analyzer/signal_information_coefficients.csv` | feature selection in every experiment after the benchmark |
| `Data/Analyzer/Charts/` | `FINDINGS_N.md` |

In [ ]:
strongest = scope_ic.reindex(scope_ic["ic"].abs().sort_values(ascending=False).index).iloc[0]

summary = pandas.DataFrame({
    "metric": [
        "refined files",
        "panel rows",
        "trading dates",
        "candidate features (ranks)",
        "eligibility column",
        f"strongest feature ({decisive_scope})",
    ],
    "value": [
        f"{len(refinery_paths)}",
        f"{len(panel):,}",
        f"{panel['m_date'].nunique():,}",
        f"{len(SIGNAL_RANK_COLUMNS)}",
        ELIGIBILITY_COLUMN or "none declared",
        f"{strongest['signal']} @ {strongest['horizon']} (IC {strongest['ic']:+.4f})",
    ],
})
print(summary.to_string(index=False))
print(f"\nCharts written to: {CHART_DIR.relative_to(REPO_ROOT)}")

---

## Open items for the Experiments stage

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **Historical sector is missing.** `sector_current` is a today-snapshot; no daily classification exists in either stage. | Sector attribution before a reclassification date is wrong. Any sector-relative feature is diagnostics-only until this closes. |
| 2 | **The Curator output is not reproducible across download dates.** Re-pulling rebases every dividend-adjusted column, because back-adjustment is computed from the present. | P&L is not exactly reproducible from a fresh download. Treat the files on disk as the record, and state the drift when you re-run. |
| 3 | **Per-ticker features may live in the Refinery for a practical reason.** Widening the Curator schema forces a full refetch of every identifier. | Purity says a per-ticker quantity is a `c_*` column; cost says compute it as `r_*` until a refetch is happening anyway. Say which you did and why. |
| 4 | **IC is measured on ranks, not on a traded portfolio.** A feature with a good IC can still lose money after costs if it decays fast (section 6.1). | Read sections 6.1 and 7 together before committing a feature to an experiment: persistence and predictive power have to both hold. |